# 07 — Skin Elasticity Regression Model

**Architecture:** EfficientNet-B0 (timm) + 14-dim handcrafted features → Fusion MLP → scalar elasticity score

**Handcrafted features (14-dim):**
- Frangi vesselness on 3 facial ROIs (forehead, left crow's feet, right crow's feet) × 3 stats = 9
- Facial landmark geometry ratios: 5

**Training:** Smooth L1 loss, AdamW lr=1e-4, CosineAnnealing, 30 epochs, batch=32

**Export:** Dual-input ONNX (image + handcrafted features)

In [ ]:
# ============================================================
# Cell 1: Install dependencies
# ============================================================
!pip install -q timm scikit-image mediapipe onnx onnxruntime

In [ ]:
# ============================================================
# Cell 2: Imports & configuration
# ============================================================
import os, json, math, warnings
import numpy as np
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

import timm
from skimage.filters import frangi
from skimage.color import rgb2gray
import mediapipe as mp

warnings.filterwarnings('ignore')

# ---------- paths ----------
DATA_ROOT   = Path('/root/cornell-hackathon/ml/data/elasticity')
IMAGES_DIR  = DATA_ROOT / 'images'
ANNOT_DIR   = DATA_ROOT / 'annotations'
OUTPUT_DIR  = Path('/root/cornell-hackathon/ml/outputs/elasticity')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- hyperparams ----------
IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_EPOCHS   = 30
LR           = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 2
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device: {DEVICE}')
print(f'Data root: {DATA_ROOT}')
print(f'Images dir exists: {IMAGES_DIR.exists()}')

In [ ]:
# ============================================================
# Cell 3: Handcrafted feature extraction (14-dim)
# ============================================================

# --- MediaPipe Face Mesh for landmark detection ---
mp_face_mesh = mp.solutions.face_mesh

# Key landmark indices (MediaPipe 468-point mesh)
# Forehead ROI center: 10 (top of forehead)
# Left eye outer: 33, Left eye inner: 133
# Right eye outer: 263, Right eye inner: 362
# Nose tip: 1, Chin: 152
# Left mouth: 61, Right mouth: 291
# Left brow outer: 46, Right brow outer: 276

FOREHEAD_IDX = 10
LEFT_EYE_OUTER = 33
LEFT_EYE_INNER = 133
RIGHT_EYE_OUTER = 263
RIGHT_EYE_INNER = 362
NOSE_TIP = 1
CHIN = 152
LEFT_MOUTH = 61
RIGHT_MOUTH = 291
LEFT_BROW_OUTER = 46
RIGHT_BROW_OUTER = 276
LEFT_CHEEK = 234
RIGHT_CHEEK = 454


def _dist(a, b):
    """Euclidean distance between two (x,y) points."""
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)


def extract_roi_frangi(gray_img, cx, cy, roi_size=48):
    """
    Extract Frangi vesselness features from a square ROI.
    Returns [mean, std, max] of Frangi response — 3 values.
    """
    h, w = gray_img.shape
    half = roi_size // 2
    y0 = max(0, cy - half)
    y1 = min(h, cy + half)
    x0 = max(0, cx - half)
    x1 = min(w, cx + half)
    roi = gray_img[y0:y1, x0:x1]
    if roi.size == 0:
        return [0.0, 0.0, 0.0]
    filt = frangi(roi, sigmas=range(1, 4), black_ridges=False)
    return [float(np.mean(filt)), float(np.std(filt)), float(np.max(filt))]


def compute_handcrafted_features(img_np):
    """
    Compute 14-dim handcrafted features from an RGB uint8 numpy image.
    Returns np.array of shape (14,) dtype float32.
    Falls back to zeros if face not detected.
    """
    h, w = img_np.shape[:2]
    gray = rgb2gray(img_np).astype(np.float64)

    features = np.zeros(14, dtype=np.float32)

    with mp_face_mesh.FaceMesh(
        static_image_mode=True,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.3
    ) as face_mesh:
        results = face_mesh.process(img_np)

    if not results.multi_face_landmarks:
        # Fallback: Frangi on center ROIs, zero landmark ratios
        features[0:3] = extract_roi_frangi(gray, w//2, h//4)       # forehead approx
        features[3:6] = extract_roi_frangi(gray, w//4, h//3)       # left crow's feet approx
        features[6:9] = extract_roi_frangi(gray, 3*w//4, h//3)     # right crow's feet approx
        return features

    lm = results.multi_face_landmarks[0].landmark

    def pt(idx):
        return (int(lm[idx].x * w), int(lm[idx].y * h))

    # --- Frangi features on 3 ROIs (9 features) ---
    # Forehead: above landmark 10
    fh = pt(FOREHEAD_IDX)
    forehead_center = (fh[0], max(0, fh[1] - int(0.05 * h)))
    features[0:3] = extract_roi_frangi(gray, forehead_center[0], forehead_center[1])

    # Left crow's feet: lateral to left eye outer corner
    le = pt(LEFT_EYE_OUTER)
    left_crow = (max(0, le[0] - int(0.03 * w)), le[1])
    features[3:6] = extract_roi_frangi(gray, left_crow[0], left_crow[1])

    # Right crow's feet: lateral to right eye outer corner
    re = pt(RIGHT_EYE_OUTER)
    right_crow = (min(w-1, re[0] + int(0.03 * w)), re[1])
    features[6:9] = extract_roi_frangi(gray, right_crow[0], right_crow[1])

    # --- Landmark geometry ratios (5 features) ---
    p_nose    = pt(NOSE_TIP)
    p_chin    = pt(CHIN)
    p_lmouth  = pt(LEFT_MOUTH)
    p_rmouth  = pt(RIGHT_MOUTH)
    p_leye_o  = pt(LEFT_EYE_OUTER)
    p_leye_i  = pt(LEFT_EYE_INNER)
    p_reye_o  = pt(RIGHT_EYE_OUTER)
    p_reye_i  = pt(RIGHT_EYE_INNER)
    p_lbrow   = pt(LEFT_BROW_OUTER)
    p_rbrow   = pt(RIGHT_BROW_OUTER)
    p_lcheek  = pt(LEFT_CHEEK)
    p_rcheek  = pt(RIGHT_CHEEK)

    face_height = _dist(fh, p_chin) + 1e-6
    eye_dist    = _dist(p_leye_i, p_reye_i) + 1e-6

    # Ratio 1: mouth width / inter-eye distance
    features[9] = _dist(p_lmouth, p_rmouth) / eye_dist
    # Ratio 2: nose-to-chin / face height
    features[10] = _dist(p_nose, p_chin) / face_height
    # Ratio 3: left eye width / inter-eye distance
    features[11] = _dist(p_leye_o, p_leye_i) / eye_dist
    # Ratio 4: right eye width / inter-eye distance
    features[12] = _dist(p_reye_o, p_reye_i) / eye_dist
    # Ratio 5: face width (cheek-to-cheek) / face height
    features[13] = _dist(p_lcheek, p_rcheek) / face_height

    return features


print('Handcrafted feature extractor ready (14-dim).')

In [ ]:
# ============================================================
# Cell 4: Dataset
# ============================================================

class ElasticityDataset(Dataset):
    def __init__(self, annotation_path, images_dir, transform=None, cache_features=False):
        with open(annotation_path) as f:
            self.records = json.load(f)
        self.images_dir = Path(images_dir)
        self.transform = transform
        self.cache_features = cache_features
        self._feature_cache = {}

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        img_path = self.images_dir / rec['image']
        img = Image.open(img_path).convert('RGB')
        img_np = np.array(img)

        # Handcrafted features (computed from original image)
        if self.cache_features and rec['image'] in self._feature_cache:
            hc_feat = self._feature_cache[rec['image']]
        else:
            hc_feat = compute_handcrafted_features(img_np)
            if self.cache_features:
                self._feature_cache[rec['image']] = hc_feat

        # Apply image transforms
        if self.transform:
            img = self.transform(img)

        target = float(rec['elasticity_score'])

        return img, torch.tensor(hc_feat, dtype=torch.float32), torch.tensor(target, dtype=torch.float32)


# ---------- transforms ----------
train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

# ---------- dataloaders ----------
train_ds = ElasticityDataset(ANNOT_DIR / 'train.json', IMAGES_DIR, transform=train_transform)
val_ds   = ElasticityDataset(ANNOT_DIR / 'val.json',   IMAGES_DIR, transform=val_transform)
test_ds  = ElasticityDataset(ANNOT_DIR / 'test.json',  IMAGES_DIR, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
# ============================================================
# Cell 5: Model definition — FusionElasticityNet
# ============================================================

class FusionElasticityNet(nn.Module):
    """
    Dual-input model:
      - Branch A: EfficientNet-B0 image encoder → 1280-dim
      - Branch B: Handcrafted features FC(14 → 32)
      - Fusion: concat(1280, 32) → FC(1312 → 256 → 1)
    """
    def __init__(self, handcrafted_dim=14, hc_hidden=32, fusion_hidden=256):
        super().__init__()

        # --- Image backbone ---
        self.backbone = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        cnn_dim = self.backbone.num_features  # 1280

        # --- Handcrafted feature branch ---
        self.hc_branch = nn.Sequential(
            nn.Linear(handcrafted_dim, hc_hidden),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(hc_hidden),
        )

        # --- Fusion head ---
        fused_dim = cnn_dim + hc_hidden  # 1312
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(fused_dim, fusion_hidden),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(fusion_hidden),
            nn.Dropout(0.2),
            nn.Linear(fusion_hidden, 1),
        )

    def forward(self, image, handcrafted):
        cnn_feat = self.backbone(image)           # (B, 1280)
        hc_feat  = self.hc_branch(handcrafted)    # (B, 32)
        fused    = torch.cat([cnn_feat, hc_feat], dim=1)  # (B, 1312)
        out      = self.head(fused).squeeze(-1)   # (B,)
        return out


model = FusionElasticityNet().to(DEVICE)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'CNN backbone features: {model.backbone.num_features}')

In [ ]:
# ============================================================
# Cell 6: Loss, optimizer, scheduler
# ============================================================

criterion = nn.SmoothL1Loss()

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

print('Loss: SmoothL1 | Optimizer: AdamW | Scheduler: CosineAnnealing')

In [ ]:
# ============================================================
# Cell 7: Training loop
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    n_samples = 0
    for images, hc_feats, targets in loader:
        images   = images.to(device, non_blocking=True)
        hc_feats = hc_feats.to(device, non_blocking=True)
        targets  = targets.to(device, non_blocking=True)

        optimizer.zero_grad()
        preds = model(images, hc_feats)
        loss = criterion(preds, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        n_samples += images.size(0)
    return running_loss / n_samples


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_targets = [], []
    n_samples = 0
    for images, hc_feats, targets in loader:
        images   = images.to(device, non_blocking=True)
        hc_feats = hc_feats.to(device, non_blocking=True)
        targets  = targets.to(device, non_blocking=True)

        preds = model(images, hc_feats)
        loss = criterion(preds, targets)

        running_loss += loss.item() * images.size(0)
        n_samples += images.size(0)
        all_preds.append(preds.cpu())
        all_targets.append(targets.cpu())

    all_preds   = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    mae = torch.mean(torch.abs(all_preds - all_targets)).item()
    return running_loss / n_samples, mae


# ---------- training ----------
best_val_mae = float('inf')
history = {'train_loss': [], 'val_loss': [], 'val_mae': [], 'lr': []}

print(f'Starting training for {NUM_EPOCHS} epochs ...')
print(f'{"Epoch":>6} {"Train Loss":>12} {"Val Loss":>12} {"Val MAE":>10} {"LR":>12}')
print('-' * 56)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_mae = evaluate(model, val_loader, criterion, DEVICE)
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_mae'].append(val_mae)
    history['lr'].append(current_lr)

    improved = ''
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        torch.save(model.state_dict(), OUTPUT_DIR / 'best_elasticity_model.pt')
        improved = ' *'

    print(f'{epoch:>6d} {train_loss:>12.5f} {val_loss:>12.5f} {val_mae:>10.3f} {current_lr:>12.2e}{improved}')

print(f'\nBest validation MAE: {best_val_mae:.3f}')

In [ ]:
# ============================================================
# Cell 8: Training curves
# ============================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss (Smooth L1)')
axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history['val_mae'], color='orange')
axes[1].set_title('Validation MAE')
axes[1].set_xlabel('Epoch')

axes[2].plot(history['lr'], color='green')
axes[2].set_title('Learning Rate')
axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()
print(f'Saved training curves to {OUTPUT_DIR / "training_curves.png"}')

In [ ]:
# ============================================================
# Cell 9: Test evaluation
# ============================================================

# Load best model
model.load_state_dict(torch.load(OUTPUT_DIR / 'best_elasticity_model.pt', map_location=DEVICE))
model.eval()

test_loss, test_mae = evaluate(model, test_loader, criterion, DEVICE)
print(f'Test Loss (SmoothL1): {test_loss:.5f}')
print(f'Test MAE:             {test_mae:.3f}')

# Collect predictions for scatter plot
all_preds, all_targets = [], []
with torch.no_grad():
    for images, hc_feats, targets in test_loader:
        images   = images.to(DEVICE)
        hc_feats = hc_feats.to(DEVICE)
        preds = model(images, hc_feats)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.numpy())

all_preds   = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

# Scatter plot
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(all_targets, all_preds, alpha=0.3, s=10)
mn, mx = min(all_targets.min(), all_preds.min()), max(all_targets.max(), all_preds.max())
ax.plot([mn, mx], [mn, mx], 'r--', label='Perfect')
ax.set_xlabel('Ground Truth Elasticity Score')
ax.set_ylabel('Predicted Elasticity Score')
ax.set_title(f'Test Set — MAE: {test_mae:.3f}')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'test_scatter.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# Cell 10: ONNX export (dual-input)
# ============================================================
import onnx

model.eval()
model_cpu = model.cpu()

dummy_image = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
dummy_hc    = torch.randn(1, 14)

onnx_path = OUTPUT_DIR / 'elasticity_model.onnx'

torch.onnx.export(
    model_cpu,
    (dummy_image, dummy_hc),
    str(onnx_path),
    input_names=['image', 'handcrafted_features'],
    output_names=['elasticity_score'],
    dynamic_axes={
        'image': {0: 'batch'},
        'handcrafted_features': {0: 'batch'},
        'elasticity_score': {0: 'batch'},
    },
    opset_version=14,
    do_constant_folding=True,
)

# Validate
onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
print(f'ONNX model exported and validated: {onnx_path}')
print(f'File size: {onnx_path.stat().st_size / 1e6:.1f} MB')

# Quick inference test with ONNX Runtime
import onnxruntime as ort
sess = ort.InferenceSession(str(onnx_path))
ort_out = sess.run(None, {
    'image': dummy_image.numpy(),
    'handcrafted_features': dummy_hc.numpy()
})
print(f'ONNX Runtime test output: {ort_out[0]}')

In [ ]:
# ============================================================
# Cell 11: Summary
# ============================================================
print('=' * 50)
print('ELASTICITY MODEL TRAINING COMPLETE')
print('=' * 50)
print(f'Best Validation MAE: {best_val_mae:.3f}')
print(f'Test MAE:            {test_mae:.3f}')
print(f'\nSaved artifacts:')
for p in sorted(OUTPUT_DIR.glob('*')):
    print(f'  {p.name:40s} ({p.stat().st_size / 1e6:.1f} MB)')